# PolicyProof AI — Native AlignTune 40-step: paired base versus trained adapter (48-case development set)

**Read first:** This notebook runs actual GPU inference, **not training**. The 48 cases were inspected during development: this is a diagnostic, not a blind evaluation. No independent human semantic validation is asserted. The existing 60-case proposed final candidate and the 72 unscored challenges are NOT used.

Your previously uploaded `policyproof_align_native40.zip` has structurally valid LoRA weights and a saved 40-step receipt. The *exact loaded model revision during training* could not be independently inferred, even though the Hugging Face revision was recorded. This notebook loads the same **specified** base commit af89d219b53ed9b13b8a4645f9c8028973510324 anew for both inference passes.

**A T4/P100 GPU, access to gated Tiny Aya Global, and your original uploaded ZIP are required.** Training is OFF and GPU inference is OFF by default. First run the preflight cells; explicitly opt into the expensive GPU step only after they pass.

## 1. Connect GPU and check out latest source. Does not load models or train.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU first."
print("GPU:", torch.cuda.get_device_name(0))
ROOT = Path("/content/PolicyProof-AI")
_prior_source = (subprocess.check_output(
    ["git","-C",str(ROOT),"rev-parse","HEAD"], text=True).strip()
    if (ROOT / ".git").is_dir() else None)
_stale_imports_at_checkout = [
    name for name in ("scripts.frozen_pair_inference", "scripts.dev48_gpu_pair")
    if name in sys.modules
]
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1",
       "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
       str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
_updated_source = subprocess.check_output(
    ["git","-C",str(ROOT),"rev-parse","HEAD"], text=True).strip()
if (_prior_source is not None and _updated_source != _prior_source
    and _stale_imports_at_checkout):
    raise RuntimeError(
        "Git source updated while older dev48 GPU Python modules were already "
        "imported. Runtime > Restart session (NOT factory reset), then "
        "rerun Stage 13 Sections 1–5. Keep existing adapter and any evidence."
    )
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
print("Source commit:",subprocess.check_output(
    ["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip())
print("NO MODEL TRAINING IN THIS CELL")


## 2. Install text-only native40 inference dependencies, safely handling optional torchvision

Colab's optional torchvision can be incompatible with its preinstalled torch, causing a misleading Transformers import error. This Tiny Aya workflow is **text-only** and does not need torchvision. The cell installs the inference dependencies without replacing torch; if torchvision is present, it removes that optional package and **requires Runtime → Restart session** before any imports or GPU work. After restart, rerun Sections 1 and 2 only. No training is performed.


In [ ]:
import subprocess, sys
from importlib.metadata import PackageNotFoundError, version
subprocess.run([sys.executable, '-m', 'pip', 'install', '--prefer-binary',
                'bitsandbytes==0.50.2', 'peft==0.21.2', 'auditkit==1.1.3'],
               check=True)
try:
    old_tv = version('torchvision')
except PackageNotFoundError:
    old_tv = None
if old_tv is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=True)
    raise RuntimeError('Optional torchvision removed from this TEXT-ONLY inference session. '
                       'RESTART THE RUNTIME (Runtime > Restart session), then rerun '
                       'Sections 1 and 2. Do NOT run GPU inference before restart.')
import torch, bitsandbytes, peft, auditkit
assert torch.cuda.is_available(), 'Choose a Colab T4 GPU runtime'
print('TEXT_ONLY_NATIVE40_INFERENCE_DEPENDENCIES_READY_NO_TRAINING')


## 3. Upload the original NATIVE ALIGNTUNE 40-step ZIP (not a checkpoint created by repacking). Verify the exact source SHA. No model inference yet.

**Upload location fix:** Colab saves `files.upload()` into the current directory (Section 1 sets it to `/content/PolicyProof-AI`). This cell searches the working directory, project directory and `/content`, selecting only a file whose **entire SHA-256 matches the original approved adapter**. If the original ZIP has already been uploaded, it will reuse it without asking you to upload again. This does not copy, rename or retrain the checkpoint.

**This notebook accepts only native AlignTune backend `aligntune_wrapper`**, with expected archive SHA256 `dcefc5b0babf708aa1b48ac0349ddb3bc4745405c72e9636c5860d7719d8f0e9`. It must not substitute the previous manual-PyTorch fallback checkpoint. Keep both models private.\n

In [ ]:
from pathlib import Path
from google.colab import files
from scripts.locate_colab_checkpoint import locate_verified_checkpoint
FILENAME="policyproof_align_native40.zip"
EXPECTED_ZIP_SHA="dcefc5b0babf708aa1b48ac0349ddb3bc4745405c72e9636c5860d7719d8f0e9"
SEARCH_DIRS=[Path.cwd(), ROOT, Path("/content")]
try:
    ARCHIVE=locate_verified_checkpoint(FILENAME, EXPECTED_ZIP_SHA, SEARCH_DIRS)
    print("Checkpoint already present; reupload not required.")
except FileNotFoundError:
    uploaded=files.upload()
    assert len(uploaded)==1 and FILENAME in uploaded, (
        "Upload exactly the ORIGINAL policyproof_align_native40.zip.")
    # Colab files.upload() saves to the CURRENT WORKING DIRECTORY.
    # Section 1 switched the current directory to ROOT, not /content.
    ARCHIVE=locate_verified_checkpoint(
        FILENAME, EXPECTED_ZIP_SHA, [Path.cwd(), ROOT, Path("/content")])
print("Original checkpoint ZIP verified:", ARCHIVE)
print("SHA-256:", EXPECTED_ZIP_SHA)


## 4. Generate exactly the original 48-case development rows and run read-only preflight.

In [ ]:
import subprocess,sys,json
subprocess.run([sys.executable,"-m","scripts.prepare"],check=True)
from scripts.dev48_gpu_pair import preflight
from scripts.check_native40_live_code import assert_live_pinned_snapshot_inference
code_status = assert_live_pinned_snapshot_inference()
print("Pinned-snapshot inference code:", code_status["status"])
assessment=preflight(ROOT,ARCHIVE,EXPECTED_ZIP_SHA,expected_backend='aligntune_wrapper')
assert assessment["status"]=="DEV48_GPU_EVALUATION_PREPARED_NOT_EXECUTED"
assert assessment["cases"]==48
print(json.dumps(assessment,indent=2,ensure_ascii=False))
print("PREFLIGHT PASSED; NO INFERENCE AND NO TRAINING PERFORMED.")


## 5. Authenticate to gated Tiny Aya without exposing the token

Use your approved Hugging Face account. If you are already logged in, this cell skips the login window. Your credentials must stay in Colab's private auth, not notebook output, GitHub or chat.
The notebook explicitly tests access to gated `config.json` with your private authenticated HF account before running any expensive GPU inference. Public revision metadata alone is insufficient.\n

In [ ]:
from scripts.restore_colab_session import verify_tiny_aya_gated_access
MODEL_ID = "CohereLabs/tiny-aya-global"
MODEL_SHA = "af89d219b53ed9b13b8a4645f9c8028973510324"
authorized_revision = verify_tiny_aya_gated_access()
assert authorized_revision == MODEL_SHA, "Different base revision; STOP before GPU inference"
print("Authorized base revision for both independent NEW inference passes:", authorized_revision)
print("No optimizer steps or model inference have run in this cell.")


## 6. GPU evaluation with cached-code guard — training remains OFF

If Section 1 fetched a newer GitHub commit after the original evaluation code had already been imported, **restart the Colab session**, not factory reset. The exact failure `Loaded Tiny Aya model did not expose an observed commit SHA` alongside the new `Path(snapshot)` source line indicates old in-memory Python functions after a git pull. Our live-code guard now checks the actual Python function objects in Sections 4 and 6 **before** fetching/loading multi-gigabyte model weights. It will stop with `STALE_STAGE13_PYTHON_CODE` if a restart is needed.

Only enable `RUN_DEV48_GPU_EVAL=True` after Sections 1–5 pass. On first run use `RESUME_SAVED_BASE=False`. If a complete baseline file exists, inspect it and use the documented resume procedure instead. The paired test contains 48 already-inspected development examples, not final independent evaluation. It does not retrain your adapter.


In [ ]:
RUN_DEV48_GPU_EVAL = False   # Change to True ONLY to run real inference
RESUME_SAVED_BASE = False   # Change to True ONLY if a complete base report exists
if RUN_DEV48_GPU_EVAL:
    from pathlib import Path
    from scripts.check_native40_live_code import assert_live_pinned_snapshot_inference
    assert_live_pinned_snapshot_inference()  # BEFORE any expensive GPU/model load
    from scripts.dev48_gpu_pair import execute_pair,create_results_bundle
    BASE=ROOT/"artifacts/native40_dev48_paired_base.json"
    TUNED=ROOT/"artifacts/native40_dev48_paired_tuned.json"
    PAIRED=ROOT/"artifacts/native40_dev48_paired_auditkit.json"
    BUNDLE=Path("/content/policyproof_native40_dev48_paired_gpu_results.zip")
    result=execute_pair(
        ROOT,ARCHIVE,EXPECTED_ZIP_SHA,BASE,TUNED,PAIRED,
        resume_base=RESUME_SAVED_BASE,
        expected_backend='aligntune_wrapper',
    )
    print("Native AuditKIT paired report status:",result["status"])
    saved=create_results_bundle(BASE,TUNED,PAIRED,BUNDLE)
    print("Evidence ZIP:",saved["archive"],saved["sha256"])
    from google.colab import files
    files.download(str(BUNDLE))
else:
    print("GPU inference OFF; model weights and optimizer were not accessed.")


## 7. CPU-only rescue if both actual prediction reports exist but AuditKIT failed

Set `RUN_CPU_AUDIT_RECOVERY=True` only if both complete **real** `BASE` and `TUNED` files exist, but no `PAIRED` file exists. This reruns only the native CPU scorer; it does not reload the model. Never substitute fabricated reports.

In [ ]:
RUN_CPU_AUDIT_RECOVERY = False
if RUN_CPU_AUDIT_RECOVERY:
    from scripts.dev48_gpu_pair import audit_existing,create_results_bundle
    from pathlib import Path
    BASE=ROOT/"artifacts/native40_dev48_paired_base.json"
    TUNED=ROOT/"artifacts/native40_dev48_paired_tuned.json"
    PAIRED=ROOT/"artifacts/native40_dev48_paired_auditkit.json"
    BUNDLE=Path("/content/policyproof_native40_dev48_paired_gpu_results.zip")
    result=audit_existing(ROOT,ARCHIVE,EXPECTED_ZIP_SHA,BASE,TUNED,PAIRED,expected_backend='aligntune_wrapper')
    print("Audit result:",result["status"])
    saved=create_results_bundle(BASE,TUNED,PAIRED,BUNDLE)
    print(saved)
    from google.colab import files
    files.download(str(BUNDLE))
else:
    print("CPU recovery OFF.")


## 8. Save results and don't call them a final benchmark

On success upload **`policyproof_native40_dev48_paired_gpu_results.zip`** to ChatGPT, keeping the original **`policyproof_align_native40.zip`** safe. The results package contains both 48-case raw prediction reports and the AuditKIT comparison, NOT model weights. We can audit individual failures and judge whether any further training is justified.

If the runtime disconnects, download whichever complete base/tuned JSON files were saved when possible. CPU and notebook mock tests are never evidence of model improvement. A genuine final result still needs two independent reviewers on the separate frozen 60-case candidate.
This is a NEW native AlignTune 40-step experiment (not the historical manual-fallback adapter). Original adapter archive SHA256: `dcefc5b0babf708aa1b48ac0349ddb3bc4745405c72e9636c5860d7719d8f0e9`. Upload the **native40** results ZIP, preserve the original checkpoint and any direct Colab run logs separately. Native AuditKIT metrics alone cannot prove semantic correctness or original training execution.
